# Classificação de Texto - Tutorial IMDB

Este notebook segue o tutorial oficial do TensorFlow [Classificação de texto com avaliações de filmes](https://www.tensorflow.org/tutorials/keras/text_classification?hl=pt-br) e utiliza o conjunto de dados **IMDB**, que reúne 50.000 avaliações de filmes em inglês, igualmente divididas entre treino e teste.

O objetivo é classificar cada avaliação como **positiva** ou **negativa**, configurando um problema de classificação binária. Ao final do notebook, o modelo treinado é salvo com a biblioteca `pickle` e é definida uma função que recebe uma nova avaliação e o modelo salvo, devolvendo o resultado da classificação.

## 1. Importação das bibliotecas

Inicialmente, são importadas as três bibliotecas básicas: o `tensorflow` (com o submódulo `keras` em destaque) para construção do modelo e o `numpy` para manipulação dos vetores numéricos. A versão do TensorFlow é exibida apenas para conferência.

In [ ]:
import tensorflow as tf
from tensorflow import keras

import numpy as np

print(tf.__version__)

## 2. Download do dataset IMDB

O conjunto de dados IMDB já vem incluído no Keras e em formato pré-processado: cada avaliação aparece como uma lista de inteiros, onde cada inteiro representa o índice de uma palavra em um dicionário. O argumento `num_words=10000` mantém apenas as 10.000 palavras mais frequentes do conjunto, descartando palavras raras.

In [ ]:
imdb = keras.datasets.imdb

(train_data, train_labels), (test_data, test_labels) = imdb.load_data(num_words=10000)

## 3. Exploração inicial dos dados

É sempre uma boa prática verificar o formato dos dados antes de começar a treinar. Aqui imprimimos o número de exemplos de treino e seus rótulos, observamos a primeira avaliação (já como sequência de inteiros) e comparamos o comprimento de duas avaliações para perceber que elas têm tamanhos diferentes.

In [ ]:
print("Training entries: {}, labels: {}".format(len(train_data), len(train_labels)))

In [ ]:
print(train_data[0])

In [ ]:
len(train_data[0]), len(train_data[1])

## 4. Convertendo inteiros de volta em palavras

Para que o conteúdo das avaliações fique legível, recuperamos o dicionário que mapeia palavras em inteiros. Os primeiros índices são reservados para tokens especiais (`<PAD>` para preenchimento, `<START>` para início de sequência, `<UNK>` para palavras desconhecidas e `<UNUSED>` para índices não utilizados). Em seguida, criamos um dicionário invertido (índice → palavra) e a função `decode_review`, que reconstrói o texto original a partir da sequência de inteiros.

In [ ]:
# Um dicionário mapeando palavras em índices inteiros
word_index = imdb.get_word_index()

# Os primeiros índices são reservados
word_index = {k: (v + 3) for k, v in word_index.items()}
word_index["<PAD>"] = 0
word_index["<START>"] = 1
word_index["<UNK>"] = 2  # unknown
word_index["<UNUSED>"] = 3

reverse_word_index = dict([(value, key) for (key, value) in word_index.items()])

def decode_review(text):
    return " ".join([reverse_word_index.get(i, "?") for i in text])

In [ ]:
decode_review(train_data[0])

## 5. Preparação dos dados

As avaliações têm comprimentos diferentes, mas as redes neurais exigem entradas de tamanho fixo. A função `pad_sequences` resolve esse problema preenchendo as sequências menores com o valor de `<PAD>` ao final (`padding='post'`) e cortando sequências maiores no limite definido por `maxlen=256`.

In [ ]:
train_data = keras.preprocessing.sequence.pad_sequences(
    train_data,
    value=word_index["<PAD>"],
    padding="post",
    maxlen=256,
)

test_data = keras.preprocessing.sequence.pad_sequences(
    test_data,
    value=word_index["<PAD>"],
    padding="post",
    maxlen=256,
)

In [ ]:
len(train_data[0]), len(train_data[1])

In [ ]:
print(train_data[0])

## 6. Construção do modelo

O modelo segue a arquitetura proposta pelo tutorial, composta por quatro camadas:

1. **Embedding**: associa cada índice de palavra a um vetor denso de dimensão 16, permitindo que o modelo aprenda representações semânticas das palavras.
2. **GlobalAveragePooling1D**: tira a média dos vetores ao longo da dimensão da sequência, devolvendo um único vetor por avaliação, independentemente do tamanho do texto.
3. **Dense(16, relu)**: camada totalmente conectada com 16 unidades e ativação ReLU, responsável por aprender combinações não lineares das representações.
4. **Dense(1, sigmoid)**: camada de saída com uma única unidade e ativação sigmoide, que devolve um valor entre 0 e 1, interpretado como a probabilidade de a avaliação ser positiva.

In [ ]:
# O formato de entrada é a contagem do vocabulário usado pelas avaliações dos filmes (10000 palavras)
vocab_size = 10000

model = keras.Sequential()
model.add(keras.layers.Embedding(vocab_size, 16))
model.add(keras.layers.GlobalAveragePooling1D())
model.add(keras.layers.Dense(16, activation="relu"))
model.add(keras.layers.Dense(1, activation="sigmoid"))

model.summary()

## 7. Compilação do modelo

Antes de treinar, o modelo precisa ser compilado. O otimizador `adam` é uma escolha sólida para a maioria dos problemas, a função de perda `binary_crossentropy` é apropriada para classificação binária com saída entre 0 e 1, e a métrica `accuracy` mede a proporção de acertos.

In [ ]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"],
)

## 8. Conjunto de validação

Para monitorar o desempenho do modelo durante o treinamento sem usar o conjunto de teste, separamos 10.000 exemplos do treino como conjunto de validação. O restante (15.000 exemplos) é usado de fato para ajustar os pesos.

In [ ]:
x_val = train_data[:10000]
partial_x_train = train_data[10000:]

y_val = train_labels[:10000]
partial_y_train = train_labels[10000:]

## 9. Treinamento do modelo

O treinamento é realizado por 40 épocas, com lotes (*batches*) de 512 exemplos. A cada época, o modelo avalia também o conjunto de validação, o que permite acompanhar se ele está aprendendo bem ou começando a sofrer com *overfitting*.

In [ ]:
history = model.fit(
    partial_x_train,
    partial_y_train,
    epochs=40,
    batch_size=512,
    validation_data=(x_val, y_val),
    verbose=1,
)

## 10. Avaliação no conjunto de teste

Após o treinamento, o desempenho final é medido no conjunto de teste, que não foi visto em nenhuma etapa anterior. Esse valor fornece uma estimativa honesta da capacidade do modelo de generalizar para novos dados.

In [ ]:
results = model.evaluate(test_data, test_labels, verbose=2)

print(results)

## 11. Histórico de treinamento

O método `fit` devolve um objeto `history` que guarda as métricas e a perda a cada época, tanto no treino quanto na validação. Esses dados permitem construir os gráficos de evolução do aprendizado.

In [ ]:
history_dict = history.history
history_dict.keys()

## 12. Gráfico de perda

Comparar a perda de treino com a de validação ajuda a identificar problemas de generalização. Quando a perda de treino continua caindo enquanto a de validação começa a subir, é sinal clássico de *overfitting*.

In [ ]:
import matplotlib.pyplot as plt

acc = history_dict["accuracy"]
val_acc = history_dict["val_accuracy"]
loss = history_dict["loss"]
val_loss = history_dict["val_loss"]

epochs = range(1, len(acc) + 1)

# "bo" significa "blue dot" (pontos azuis)
plt.plot(epochs, loss, "bo", label="Training loss")
# "b" significa linha azul contínua
plt.plot(epochs, val_loss, "b", label="Validation loss")
plt.title("Training and validation loss")
plt.xlabel("Epochs")
plt.ylabel("Loss")
plt.legend()

plt.show()

## 13. Gráfico de acurácia

Da mesma forma, comparar a acurácia de treino com a de validação ao longo das épocas mostra como o modelo está evoluindo em termos de proporção de acertos.

In [ ]:
plt.clf()  # limpa a figura anterior

plt.plot(epochs, acc, "bo", label="Training acc")
plt.plot(epochs, val_acc, "b", label="Validation acc")
plt.title("Training and validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("Accuracy")
plt.legend()

plt.show()

## 14. Exportação do modelo com `pickle`

O enunciado pede que o modelo treinado seja salvo com a biblioteca `pickle`. Como modelos Keras não são serializáveis diretamente por `pickle` de forma confiável, salvamos o modelo em formato Keras nativo (`.keras`) e armazenamos com `pickle` um *bundle* contendo o caminho do modelo e o dicionário de palavras, necessário para preparar novas entradas.

In [ ]:
import os
import pickle

MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

keras_model_path = os.path.join(MODELS_DIR, "imdb_model.keras")
pickle_path = os.path.join(MODELS_DIR, "imdb_model.pkl")

model.save(keras_model_path)

bundle = {
    "keras_model_path": keras_model_path,
    "word_index": word_index,
    "maxlen": 256,
    "label_mapping": {0: "negativa", 1: "positiva"},
}

with open(pickle_path, "wb") as f:
    pickle.dump(bundle, f)

print("Modelo salvo em:", keras_model_path)
print("Bundle pickle salvo em:", pickle_path)

## 15. Função de classificação para novos textos

A função `classificar_avaliacao` recebe um texto em inglês e o caminho do `pickle` salvo. Internamente, ela transforma o texto em uma sequência de inteiros usando o mesmo `word_index` aprendido durante o treinamento, aplica *padding* até `maxlen=256` e usa o modelo carregado para prever a classe. O resultado é um dicionário com a probabilidade estimada e a classe predita.

In [ ]:
def classificar_avaliacao(texto, pickle_path="../models/imdb_model.pkl"):
    with open(pickle_path, "rb") as f:
        b = pickle.load(f)

    modelo = keras.models.load_model(b["keras_model_path"])
    word_index_local = b["word_index"]
    maxlen = b["maxlen"]

    # Converte o texto em uma sequência de inteiros (com <START> no início)
    tokens = texto.lower().split()
    sequencia = [1] + [word_index_local.get(t, 2) for t in tokens]
    sequencia = keras.preprocessing.sequence.pad_sequences(
        [sequencia], value=0, padding="post", maxlen=maxlen
    )

    probabilidade = float(modelo.predict(sequencia, verbose=0)[0][0])
    classe = b["label_mapping"][1 if probabilidade >= 0.5 else 0]

    return {
        "texto": texto,
        "probabilidade_positiva": probabilidade,
        "classe_predita": classe,
    }

In [ ]:
exemplos = [
    "the movie was absolutely wonderful and i loved every minute of it",
    "what a waste of time the plot was terrible and the acting was worse",
    "it was an average film nothing particularly remarkable",
]

for texto in exemplos:
    print(classificar_avaliacao(texto))

## 16. Considerações finais

Este notebook reproduziu o tutorial de classificação de texto do TensorFlow usando o dataset IMDB já pré-tokenizado. As etapas seguiram exatamente a estrutura proposta: importação das bibliotecas básicas, carregamento dos dados, exploração, preparação com *padding*, construção e compilação do modelo, treinamento por 40 épocas com conjunto de validação, avaliação no conjunto de teste e visualização dos gráficos de perda e acurácia.

Para atender ao enunciado, foram adicionadas duas etapas extras: a exportação do modelo treinado via `pickle` e a função `classificar_avaliacao`, que recebe um novo texto e o modelo salvo e devolve o resultado da classificação.